# UFC Silver Layer

Cleans and standardizes the bronze tables into `ufc_silver`: parsed event dates and locations, resolved winner names, fighter metrics (BMI, win rate, weight class), and fight durations. Depends on bronze tables produced by the **01_bronze_ingestion** notebook.

In [0]:
# ============================================================
# SILVER LAYER — Setup
# ============================================================
from pyspark.sql.functions import *
from pyspark.sql.types import *

SILVER_SCHEMA = "ufc_silver"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
spark.sql(f"COMMENT ON SCHEMA {SILVER_SCHEMA} IS 'Silver layer: cleaned and standardized UFC data'")

def bronze_table_exists(table_name):
    try:
        rows = spark.sql("SHOW TABLES IN ufc_bronze").collect()
        return any(r.tableName == table_name for r in rows)
    except Exception:
        return False

print("Silver setup complete. Bronze tables available:")
display(spark.sql("SHOW TABLES IN ufc_bronze"))

In [0]:
# --- Silver: Events (from completed_events_small) ---
if bronze_table_exists("completed_events_small"):
    print("=== Silver: completed_events_small -> events ===")
    df = spark.read.table("ufc_bronze.completed_events_small")
    df = df.drop("_source_file", "_ingestion_timestamp").dropDuplicates()
    for c in df.columns:
        if dict(df.dtypes)[c] == "string":
            df = df.withColumn(c, trim(col(c)))
    if "date" in df.columns:
        df = (df.withColumn("date",
              coalesce(to_date(col("date"), "MMMM d, yyyy"),
                       to_date(col("date"), "yyyy-MM-dd"),
                       to_date(col("date"), "MM/dd/yyyy")))
              .withColumn("event_year", year(col("date")))
              .withColumn("event_month", month(col("date")))
              .withColumn("event_quarter", quarter(col("date"))))
    if "location" in df.columns:
        loc_parts = split(col("location"), ",")
        df = (df
              .withColumn("city", trim(loc_parts.getItem(0)))
              .withColumn("state", when(size(loc_parts) >= 3, trim(loc_parts.getItem(1))).otherwise(lit(None)))
              .withColumn("country", when(size(loc_parts) >= 3, trim(loc_parts.getItem(2))).otherwise(trim(loc_parts.getItem(1)))))
    df.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER_SCHEMA}.events")
    print(f"  Rows: {df.count()}")
    display(df.limit(5))
    print(f"  ✅ {SILVER_SCHEMA}.events created!")
else:
    print("⚠️  completed_events_small not found")

In [0]:
# --- Silver: Fights (from large_dataset — main 95-col fight table) ---
if bronze_table_exists("large_dataset"):
    print("=== Silver: large_dataset -> fights ===")
    df = spark.read.table("ufc_bronze.large_dataset")
    df = df.drop("_source_file", "_ingestion_timestamp").dropDuplicates()
    for c in df.columns:
        if dict(df.dtypes)[c] == "string":
            df = df.withColumn(c, trim(col(c)))
    # Resolve winner from "Red"/"Blue" to fighter name
    if "winner" in df.columns and "r_fighter" in df.columns and "b_fighter" in df.columns:
        df = (df.withColumn("winner_name",
              when(col("winner") == "Red", col("r_fighter"))
              .when(col("winner") == "Blue", col("b_fighter"))
              .otherwise(col("winner")))
              .withColumn("winner_is_red",
              when(col("winner") == "Red", True)
              .when(col("winner") == "Blue", False)
              .otherwise(None)))
    # Standardize finish category
    if "method" in df.columns:
        df = (df.withColumn("finish_category",
              when(col("method").rlike("(?i)KO|TKO"), "KO/TKO")
              .when(col("method").rlike("(?i)sub|submission"), "Submission")
              .when(col("method").rlike("(?i)dec|decision"), "Decision")
              .when(col("method").rlike("(?i)DQ|disqual"), "DQ")
              .otherwise(col("method"))))
    df.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER_SCHEMA}.fights")
    print(f"  Rows: {df.count()}")
    display(df.select("event_name", "r_fighter", "b_fighter", "winner_name", "finish_category", "method", "finish_round", "time_sec").limit(5))
    print(f"  ✅ {SILVER_SCHEMA}.fights created!")
else:
    print("⚠️  large_dataset not found")

In [0]:
# --- Silver: Fighters (from fighter_stats) ---
if bronze_table_exists("fighter_stats"):
    print("=== Silver: fighter_stats -> fighters ===")
    df = spark.read.table("ufc_bronze.fighter_stats")
    df = df.drop("_source_file", "_ingestion_timestamp").dropDuplicates()
    for c in df.columns:
        if dict(df.dtypes)[c] == "string":
            df = df.withColumn(c, trim(col(c)))
    # Rename for clarity
    if "height" in df.columns:
        df = df.withColumnRenamed("height", "height_cm")
    if "weight" in df.columns:
        df = df.withColumnRenamed("weight", "weight_kg")
    if "reach" in df.columns:
        df = df.withColumnRenamed("reach", "reach_cm")
    # BMI, win_rate, weight_class
    if "height_cm" in df.columns and "weight_kg" in df.columns:
        df = (df.withColumn("bmi",
              when(col("height_cm").isNotNull() & col("weight_kg").isNotNull() & (col("height_cm") > 0),
                   round(col("weight_kg") / (col("height_cm") / 100) ** 2, 2))
              .otherwise(None)))
    if "wins" in df.columns and "losses" in df.columns:
        df = (df.withColumn("win_rate",
              when((col("wins") + col("losses")) > 0,
                   round(col("wins") / (col("wins") + col("losses")), 4))
              .otherwise(None)))
    if "weight_kg" in df.columns:
        df = (df.withColumn("weight_class",
              when(col("weight_kg") <= 56.7, "Flyweight")
              .when(col("weight_kg") <= 61.2, "Bantamweight")
              .when(col("weight_kg") <= 65.8, "Featherweight")
              .when(col("weight_kg") <= 70.3, "Lightweight")
              .when(col("weight_kg") <= 77.1, "Welterweight")
              .when(col("weight_kg") <= 83.9, "Middleweight")
              .when(col("weight_kg") <= 93.0, "Light Heavyweight")
              .when(col("weight_kg") > 93.0, "Heavyweight")
              .otherwise(None)))
    df.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER_SCHEMA}.fighters")
    print(f"  Rows: {df.count()}")
    display(df.select("name", "wins", "losses", "height_cm", "weight_kg", "reach_cm", "stance", "age", "bmi", "win_rate", "weight_class").limit(5))
    print(f"  ✅ {SILVER_SCHEMA}.fighters created!")
else:
    print("⚠️  fighter_stats not found")

In [0]:
# --- Silver: Fights Medium (from medium_dataset — alternative fight stats) ---
if bronze_table_exists("medium_dataset"):
    print("=== Silver: medium_dataset -> fights_medium ===")
    df = spark.read.table("ufc_bronze.medium_dataset")
    df = df.drop("_source_file", "_ingestion_timestamp").dropDuplicates()
    for c in df.columns:
        if dict(df.dtypes)[c] == "string":
            df = df.withColumn(c, trim(col(c)))
    if "date" in df.columns:
        df = (df.withColumn("date",
              coalesce(to_date(col("date"), "M/d/yyyy"),
                       to_date(col("date"), "MMMM d, yyyy"),
                       to_date(col("date"), "yyyy-MM-dd")))
              .withColumn("event_year", year(col("date")))
              .withColumn("event_month", month(col("date"))))
    if "location" in df.columns:
        loc_parts = split(col("location"), ",")
        df = (df
              .withColumn("city", trim(loc_parts.getItem(0)))
              .withColumn("state", when(size(loc_parts) >= 3, trim(loc_parts.getItem(1))).otherwise(lit(None)))
              .withColumn("country", when(size(loc_parts) >= 3, trim(loc_parts.getItem(2))).otherwise(trim(loc_parts.getItem(1)))))
    if "round" in df.columns:
        df = df.withColumn("round", col("round").cast("int"))
    if "time" in df.columns:
        time_parts = split(col("time"), ":")
        df = (df.withColumn("fight_duration_seconds",
              when(size(time_parts) == 2,
                   time_parts.getItem(0).cast("int") * 60 + time_parts.getItem(1).cast("int"))
              .otherwise(col("time").cast("int"))))
    df.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER_SCHEMA}.fights_medium")
    print(f"  Rows: {df.count()}")
    display(df.select("event", "date", "r_fighter", "b_fighter", "method", "round", "fight_duration_seconds").limit(5))
    print(f"  ✅ {SILVER_SCHEMA}.fights_medium created!")
else:
    print("⚠️  medium_dataset not found")